# 03 - Synthetic-Real Feature Quality Diagnostic

This notebook asks whether poor real-data recovery is plausibly caused by a mismatch between simulator-generated observed features and real baseline observed features.

For each configured feature set it compares synthetic observations against real holdout observations, runs classifier two-sample tests, loads saved recovery metrics if available, and writes a timestamped diagnostic folder.

The central classifier test uses labels `synthetic = 0` and `real = 1`. High AUC means the simulator and real feature distributions are easy to distinguish.


## Setup

In [ ]:
from pathlib import Path
from datetime import datetime
import copy
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import ks_2samp, wasserstein_distance
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SRC_CANDIDATES = [Path("../src"), Path("biological_age_sbi/experiment/src")]
for src in SRC_CANDIDATES:
    if (src / "bioage_sbi").exists():
        sys.path.insert(0, str(src.resolve()))
        break
else:
    raise FileNotFoundError("Could not find biological_age_sbi/experiment/src")

from bioage_sbi import (
    FEATURE_SETS,
    find_baseline_data_path,
    fit_empirical_model,
    load_baseline_data,
    prepare_model_frame,
    save_empirical_model,
    split_train_holdout,
)
from bioage_sbi.simulator import sample_component_model

sns.set_theme(style="whitegrid")
np.set_printoptions(suppress=True)


## Configuration

In [ ]:
SEED = 1234
SPLIT_SEED = 2026
SIMULATION_SEED_BASE = SEED + 100
CLASSIFIER_SEED_BASE = SEED + 200
PCA_SEED_BASE = SEED + 300

FEATURE_SET_NAMES = [
    "set_a_easy_non_lab",
    "set_f_common_non_lab_diagnosis",
    "set_e_kdm8_common_lab",
]
SIMULATOR_VARIANT = "copula_empirical_residuals_sbp_bmi_0_5_sbp_agebin_mean_0_5"

COPULA_ENABLED = True
CONTINUOUS_RESIDUAL_NOISE_SCALE = 1.0
OBSERVATION_NOISE_SCALE = 1.0
LATENT_STD_SCALE = 1.0
LATENT_AGE_EFFECT_SCALE = 1.0
LATENT_LOADING_SCALE = 1.0
CONTINUOUS_EMPIRICAL_RESIDUAL_BOOTSTRAP = True
SBP_BMI_EFFECT_SCALE = 0.5
SBP_AGE_BIN_MEAN_CORRECTION = True
SBP_AGE_BIN_MEAN_CORRECTION_SCALE = 0.5
AGE_BIN_CALIBRATION_SYNTHETIC_N = None  # None means match train size.

N_SYNTHETIC_PER_FEATURE_SET = None  # None means match holdout size.
MAX_DOMAIN_CLASSIFIER_PER_CLASS = 3000
DOMAIN_TEST_SIZE = 0.30
TOP_N_MISMATCH_FEATURES = 6
TOP_N_CLASSIFIER_FEATURES = 10
AGE_BINS = np.arange(40, 101, 10)

GOOD_SYNTHETIC_R = 0.70
POOR_R = 0.30
ACCEPTABLE_REAL_R = 0.50

RUN_UMAP = True
ALLOW_RECOVERY_RETRAINING = False

DATA_PATH = find_baseline_data_path()
EXPERIMENT_DIR = DATA_PATH.parents[2]
PROCESSED_DIR = DATA_PATH.parent
FEATURE_QUALITY_ROOT = EXPERIMENT_DIR / "results" / "data_diagnostics" / "synthetic_feature_quality"
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = FEATURE_QUALITY_ROOT / TIMESTAMP
PLOTS_DIR = OUTPUT_DIR / "plots"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR


## Load Data And Fit Feature-Set Simulators

In [ ]:
def apply_simulator_variant(model):
    active_model = copy.deepcopy(model)

    for spec in active_model["continuous_models"].values():
        spec["residual_std"] = float(spec["residual_std"] * CONTINUOUS_RESIDUAL_NOISE_SCALE)

    continuous_noise = active_model["observation_model"]["continuous_noise_std"]
    for key, value in continuous_noise.items():
        continuous_noise[key] = float(value * OBSERVATION_NOISE_SCALE)

    dependence_model = active_model.get("dependence_model")
    if dependence_model is not None:
        dependence_model["enabled"] = bool(COPULA_ENABLED)

    latent_config = active_model.get("latent_factors", {})
    for spec in latent_config.get("factors", {}).values():
        spec["std"] = float(spec["std"] * LATENT_STD_SCALE)
        if "age_z" in spec.get("mean_terms", {}):
            spec["mean_terms"]["age_z"] = float(spec["mean_terms"]["age_z"] * LATENT_AGE_EFFECT_SCALE)

    for loadings_by_variable in [
        latent_config.get("continuous_loadings", {}),
        latent_config.get("binary_logit_loadings", {}),
    ]:
        for loadings in loadings_by_variable.values():
            for key, value in loadings.items():
                loadings[key] = float(value * LATENT_LOADING_SCALE)

    continuous_calibration = active_model.get("calibration", {}).get("continuous")
    if continuous_calibration is not None:
        continuous_calibration["enabled"] = bool(CONTINUOUS_EMPIRICAL_RESIDUAL_BOOTSTRAP)
        continuous_calibration["empirical_residual_bootstrap_enabled"] = bool(CONTINUOUS_EMPIRICAL_RESIDUAL_BOOTSTRAP)
        continuous_calibration.setdefault("effect_scales", {})["sbp_bmi"] = float(SBP_BMI_EFFECT_SCALE)

    active_model["simulator_variant"] = {
        "name": SIMULATOR_VARIANT,
        "copula_enabled": bool(active_model.get("dependence_model", {}).get("enabled", False)),
        "continuous_residual_noise_scale": CONTINUOUS_RESIDUAL_NOISE_SCALE,
        "observation_noise_scale": OBSERVATION_NOISE_SCALE,
        "latent_std_scale": LATENT_STD_SCALE,
        "latent_age_effect_scale": LATENT_AGE_EFFECT_SCALE,
        "latent_loading_scale": LATENT_LOADING_SCALE,
        "continuous_empirical_residual_bootstrap": bool(CONTINUOUS_EMPIRICAL_RESIDUAL_BOOTSTRAP),
        "sbp_bmi_effect_scale": SBP_BMI_EFFECT_SCALE,
        "sbp_age_bin_mean_correction": bool(SBP_AGE_BIN_MEAN_CORRECTION),
        "sbp_age_bin_mean_correction_scale": SBP_AGE_BIN_MEAN_CORRECTION_SCALE,
    }
    return active_model


def synthetic_observed_frame(model, num_samples, seed):
    simulated = sample_component_model(model, num_samples=num_samples, seed=seed)
    observed_columns = [model["observed_key_by_column"][col] for col in model["columns"]]
    rename = {model["observed_key_by_column"][col]: col for col in model["columns"]}
    return simulated[["biological_age", *observed_columns]].rename(columns=rename)


def real_observed_frame(model, holdout_frame):
    return holdout_frame[["biological_age", *model["columns"]]].copy()


def calibrate_sbp_age_bin_mean(model, train_frame, seed):
    if not SBP_AGE_BIN_MEAN_CORRECTION or "sbp" not in model["columns"]:
        return model

    continuous_calibration = model.get("calibration", {}).get("continuous")
    if continuous_calibration is None:
        return model

    n_synthetic = len(train_frame) if AGE_BIN_CALIBRATION_SYNTHETIC_N is None else AGE_BIN_CALIBRATION_SYNTHETIC_N
    synthetic_train = synthetic_observed_frame(model, n_synthetic, seed=seed)
    real_train = train_frame[["biological_age", "sbp"]].copy()

    synthetic_bins = pd.cut(synthetic_train["biological_age"], bins=AGE_BINS, include_lowest=True)
    real_bins = pd.cut(real_train["biological_age"], bins=AGE_BINS, include_lowest=True)
    adjustments = []
    for interval in synthetic_bins.cat.categories:
        synthetic_mean = synthetic_train.loc[synthetic_bins == interval, "sbp"].mean()
        real_mean = real_train.loc[real_bins == interval, "sbp"].mean()
        if np.isfinite(synthetic_mean) and np.isfinite(real_mean):
            adjustment = (real_mean - synthetic_mean) * SBP_AGE_BIN_MEAN_CORRECTION_SCALE
        else:
            adjustment = 0.0
        adjustments.append(float(adjustment))

    age_adjustment = continuous_calibration.setdefault("age_bin_mean_adjustment", {})
    age_adjustment["enabled"] = True
    age_adjustment.setdefault("columns", {})["sbp"] = adjustments
    age_adjustment["source"] = "train_split_synthetic_minus_real_age_bin_mean"
    age_adjustment["scale"] = float(SBP_AGE_BIN_MEAN_CORRECTION_SCALE)
    age_adjustment["synthetic_n"] = int(n_synthetic)
    return model


raw_df = load_baseline_data(DATA_PATH)
feature_set_artifacts = {}
feature_set_summary_rows = []

for feature_set_name in FEATURE_SET_NAMES:
    model_frame = prepare_model_frame(raw_df, feature_set_name=feature_set_name)
    train_frame, holdout_frame = split_train_holdout(model_frame, holdout_fraction=0.20, seed=SPLIT_SEED)
    empirical_model = fit_empirical_model(
        train_frame,
        model_name=feature_set_name,
        feature_set_name=feature_set_name,
    )
    active_model = apply_simulator_variant(empirical_model)
    active_model = calibrate_sbp_age_bin_mean(
        active_model,
        train_frame,
        seed=SIMULATION_SEED_BASE + 10_000 + len(feature_set_artifacts),
    )

    model_path = PROCESSED_DIR / f"empirical_bioage_model_{feature_set_name}.json"
    train_path = PROCESSED_DIR / f"baseline_train_{feature_set_name}.csv"
    holdout_path = PROCESSED_DIR / f"baseline_holdout_{feature_set_name}.csv"
    save_empirical_model(empirical_model, model_path)
    train_frame.to_csv(train_path, index=False)
    holdout_frame.to_csv(holdout_path, index=False)

    n_synthetic = len(holdout_frame) if N_SYNTHETIC_PER_FEATURE_SET is None else N_SYNTHETIC_PER_FEATURE_SET
    synthetic_frame = synthetic_observed_frame(active_model, n_synthetic, seed=SIMULATION_SEED_BASE + len(feature_set_artifacts))
    real_frame = real_observed_frame(active_model, holdout_frame)

    feature_set_artifacts[feature_set_name] = {
        "model": active_model,
        "train_frame": train_frame,
        "holdout_frame": holdout_frame,
        "synthetic_frame": synthetic_frame,
        "real_frame": real_frame,
        "model_path": model_path,
    }
    feature_set_summary_rows.append(
        {
            "feature_set_name": feature_set_name,
            "number_of_features": len(active_model["columns"]),
            "selected_feature_names": ", ".join(active_model["columns"]),
            "n_synthetic": len(synthetic_frame),
            "n_real": len(real_frame),
            "feature_set_description": active_model["feature_set_description"],
        }
    )

feature_set_summary = pd.DataFrame(feature_set_summary_rows)
feature_set_summary


## Feature Mismatch Metrics

In [ ]:
def age_matched_frames(sim_frame, real_frame, seed=SEED):
    rng = np.random.default_rng(seed)
    sim = sim_frame.copy()
    real = real_frame.copy()
    sim["age_bin"] = pd.cut(sim["biological_age"], bins=AGE_BINS, include_lowest=True)
    real["age_bin"] = pd.cut(real["biological_age"], bins=AGE_BINS, include_lowest=True)

    sim_parts = []
    real_parts = []
    for age_bin in sorted(set(sim["age_bin"].dropna()) & set(real["age_bin"].dropna())):
        sim_bin = sim[sim["age_bin"] == age_bin]
        real_bin = real[real["age_bin"] == age_bin]
        n = min(len(sim_bin), len(real_bin))
        if n == 0:
            continue
        sim_parts.append(sim_bin.sample(n=n, random_state=int(rng.integers(0, 1_000_000))))
        real_parts.append(real_bin.sample(n=n, random_state=int(rng.integers(0, 1_000_000))))

    if not sim_parts or not real_parts:
        return sim_frame.iloc[0:0].copy(), real_frame.iloc[0:0].copy()
    return (
        pd.concat(sim_parts, ignore_index=True).drop(columns=["age_bin"]),
        pd.concat(real_parts, ignore_index=True).drop(columns=["age_bin"]),
    )


def compute_feature_mismatch(feature_set_name, sim_frame, real_frame, feature_columns, comparison_type):
    rows = []
    for feature in feature_columns:
        sim_values = pd.to_numeric(sim_frame[feature], errors="coerce")
        real_values = pd.to_numeric(real_frame[feature], errors="coerce")
        sim_clean = sim_values.dropna().to_numpy(dtype=float)
        real_clean = real_values.dropna().to_numpy(dtype=float)

        mean_sim = float(np.nanmean(sim_values))
        mean_real = float(np.nanmean(real_values))
        std_sim = float(np.nanstd(sim_values, ddof=1))
        std_real = float(np.nanstd(real_values, ddof=1))
        pooled_std = float(np.sqrt((std_sim**2 + std_real**2) / 2.0))
        smd = np.nan if pooled_std < 1e-12 else (mean_sim - mean_real) / pooled_std
        wasserstein = wasserstein_distance(sim_clean, real_clean) if len(sim_clean) and len(real_clean) else np.nan
        is_continuous_like = max(pd.Series(sim_clean).nunique(), pd.Series(real_clean).nunique()) > 2
        ks_stat = ks_2samp(sim_clean, real_clean).statistic if is_continuous_like and len(sim_clean) and len(real_clean) else np.nan

        rows.append(
            {
                "feature_set_name": feature_set_name,
                "comparison_type": comparison_type,
                "feature": feature,
                "mean_sim": mean_sim,
                "mean_real": mean_real,
                "std_sim": std_sim,
                "std_real": std_real,
                "standardized_mean_difference": smd,
                "absolute_standardized_mean_difference": abs(smd) if pd.notna(smd) else np.nan,
                "wasserstein_distance": float(wasserstein),
                "ks_statistic": float(ks_stat) if pd.notna(ks_stat) else np.nan,
                "missing_rate_sim": float(sim_values.isna().mean()),
                "missing_rate_real": float(real_values.isna().mean()),
                "missing_rate_difference": float(sim_values.isna().mean() - real_values.isna().mean()),
                "n_sim": int(len(sim_values)),
                "n_real": int(len(real_values)),
            }
        )
    return pd.DataFrame(rows)


feature_mismatch_tables = []
for feature_set_name, artifact in feature_set_artifacts.items():
    model = artifact["model"]
    feature_columns = model["columns"]
    sim_frame = artifact["synthetic_frame"]
    real_frame = artifact["real_frame"]

    feature_mismatch_tables.append(
        compute_feature_mismatch(feature_set_name, sim_frame, real_frame, feature_columns, "unconditional")
    )
    sim_age_matched, real_age_matched = age_matched_frames(sim_frame, real_frame, seed=SEED)
    if len(sim_age_matched) and len(real_age_matched):
        feature_mismatch_tables.append(
            compute_feature_mismatch(feature_set_name, sim_age_matched, real_age_matched, feature_columns, "age_matched")
        )

feature_mismatch_metrics = pd.concat(feature_mismatch_tables, ignore_index=True)
feature_mismatch_metrics = feature_mismatch_metrics.sort_values(
    ["feature_set_name", "comparison_type", "absolute_standardized_mean_difference"],
    ascending=[True, True, False],
)
feature_mismatch_metrics.to_csv(OUTPUT_DIR / "feature_mismatch_metrics.csv", index=False)
feature_mismatch_metrics.head(12)


## Correlation Mismatch Metrics And Heatmaps

In [ ]:
def plot_correlation_heatmap(matrix, title, path, vmin=-1.0, vmax=1.0):
    fig, ax = plt.subplots(figsize=(max(5, 0.45 * len(matrix.columns)), max(4, 0.45 * len(matrix.columns))))
    sns.heatmap(matrix, cmap="vlag", center=0, vmin=vmin, vmax=vmax, ax=ax)
    ax.set_title(title)
    plt.tight_layout()
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)


correlation_rows = []
for feature_set_name, artifact in feature_set_artifacts.items():
    feature_columns = artifact["model"]["columns"]
    sim_corr = artifact["synthetic_frame"][feature_columns].corr()
    real_corr = artifact["real_frame"][feature_columns].corr()
    corr_diff = sim_corr - real_corr

    plot_correlation_heatmap(sim_corr, f"Synthetic correlations: {feature_set_name}", PLOTS_DIR / f"correlation_synthetic_{feature_set_name}.png")
    plot_correlation_heatmap(real_corr, f"Real correlations: {feature_set_name}", PLOTS_DIR / f"correlation_real_{feature_set_name}.png")
    plot_correlation_heatmap(corr_diff, f"Synthetic - real correlations: {feature_set_name}", PLOTS_DIR / f"correlation_difference_{feature_set_name}.png")

    pair_abs_diffs = []
    for i, feature_i in enumerate(feature_columns):
        for feature_j in feature_columns[i + 1:]:
            diff = float(corr_diff.loc[feature_i, feature_j])
            pair_abs_diffs.append(abs(diff))
            correlation_rows.append(
                {
                    "feature_set_name": feature_set_name,
                    "feature_i": feature_i,
                    "feature_j": feature_j,
                    "corr_sim": float(sim_corr.loc[feature_i, feature_j]),
                    "corr_real": float(real_corr.loc[feature_i, feature_j]),
                    "correlation_difference": diff,
                    "absolute_correlation_difference": abs(diff),
                    "mean_absolute_correlation_difference": np.nan,
                    "max_absolute_correlation_difference": np.nan,
                }
            )

    feature_set_mask = [row["feature_set_name"] == feature_set_name for row in correlation_rows]
    mean_abs = float(np.nanmean(pair_abs_diffs)) if pair_abs_diffs else np.nan
    max_abs = float(np.nanmax(pair_abs_diffs)) if pair_abs_diffs else np.nan
    for row, is_current in zip(correlation_rows, feature_set_mask):
        if is_current:
            row["mean_absolute_correlation_difference"] = mean_abs
            row["max_absolute_correlation_difference"] = max_abs

correlation_mismatch_metrics = pd.DataFrame(correlation_rows).sort_values(
    ["feature_set_name", "absolute_correlation_difference"],
    ascending=[True, False],
)
correlation_mismatch_metrics.to_csv(OUTPUT_DIR / "correlation_mismatch_metrics.csv", index=False)
correlation_mismatch_metrics.head(12)


## Continuous Calibration Targets: Shape, Age Structure, And Joint Dependence

BMI and SBP are continuous features and need more targeted diagnostics than marginal mean/SD alone. This section focuses on three calibration targets:

1. Distribution shape: quantiles, skewness, and clinical tail rates.
2. Age-conditional structure: mean, SD, quantiles, and tail rates by biological-age bin.
3. Joint dependence: BMI-SBP correlation overall and by age bin.


In [ ]:
CALIBRATION_CONTINUOUS_FEATURES = ["bmi", "sbp", "platelets", "crp", "hba1c", "creatinine", "bun", "tc", "tg"]
SHAPE_QUANTILES = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
TAIL_DEFINITIONS = {
    "bmi": {
        "lt_18_5": ("lt", 18.5),
        "ge_30": ("ge", 30.0),
        "ge_35": ("ge", 35.0),
    },
    "sbp": {
        "lt_90": ("lt", 90.0),
        "ge_140": ("ge", 140.0),
        "ge_160": ("ge", 160.0),
    },
}


def target_continuous_features(model):
    return [feature for feature in CALIBRATION_CONTINUOUS_FEATURES if feature in model["continuous_columns"]]


def tail_rate(values, operator, threshold):
    values = pd.to_numeric(pd.Series(values), errors="coerce").dropna().to_numpy(dtype=float)
    if len(values) == 0:
        return np.nan
    if operator == "lt":
        return float(np.mean(values < threshold))
    if operator == "le":
        return float(np.mean(values <= threshold))
    if operator == "gt":
        return float(np.mean(values > threshold))
    if operator == "ge":
        return float(np.mean(values >= threshold))
    raise ValueError(f"Unknown operator: {operator}")


def numeric_clean(frame, feature):
    return pd.to_numeric(frame[feature], errors="coerce").dropna().to_numpy(dtype=float)


def compute_shape_metrics(feature_set_name, sim_frame, real_frame, feature):
    sim_values = numeric_clean(sim_frame, feature)
    real_values = numeric_clean(real_frame, feature)
    row = {
        "feature_set_name": feature_set_name,
        "feature": feature,
        "n_sim": len(sim_values),
        "n_real": len(real_values),
        "mean_sim": float(np.mean(sim_values)),
        "mean_real": float(np.mean(real_values)),
        "mean_difference": float(np.mean(sim_values) - np.mean(real_values)),
        "std_sim": float(np.std(sim_values, ddof=1)),
        "std_real": float(np.std(real_values, ddof=1)),
        "std_ratio_sim_to_real": float(np.std(sim_values, ddof=1) / np.std(real_values, ddof=1)),
        "skew_sim": float(pd.Series(sim_values).skew()),
        "skew_real": float(pd.Series(real_values).skew()),
        "skew_difference": float(pd.Series(sim_values).skew() - pd.Series(real_values).skew()),
    }
    for q in SHAPE_QUANTILES:
        label = f"q{int(q * 100):02d}"
        sim_q = float(np.quantile(sim_values, q))
        real_q = float(np.quantile(real_values, q))
        row[f"{label}_sim"] = sim_q
        row[f"{label}_real"] = real_q
        row[f"{label}_difference"] = sim_q - real_q

    for tail_name, (operator, threshold) in TAIL_DEFINITIONS.get(feature, {}).items():
        sim_tail = tail_rate(sim_values, operator, threshold)
        real_tail = tail_rate(real_values, operator, threshold)
        row[f"tail_{tail_name}_sim"] = sim_tail
        row[f"tail_{tail_name}_real"] = real_tail
        row[f"tail_{tail_name}_difference"] = sim_tail - real_tail
    return row


def compute_age_conditional_metrics(feature_set_name, sim_frame, real_frame, feature):
    sim = sim_frame[["biological_age", feature]].copy()
    real = real_frame[["biological_age", feature]].copy()
    sim["age_bin"] = pd.cut(sim["biological_age"], bins=AGE_BINS, include_lowest=True)
    real["age_bin"] = pd.cut(real["biological_age"], bins=AGE_BINS, include_lowest=True)
    rows = []
    for age_bin in sorted(set(sim["age_bin"].dropna()) | set(real["age_bin"].dropna())):
        sim_values = pd.to_numeric(sim.loc[sim["age_bin"] == age_bin, feature], errors="coerce").dropna().to_numpy(dtype=float)
        real_values = pd.to_numeric(real.loc[real["age_bin"] == age_bin, feature], errors="coerce").dropna().to_numpy(dtype=float)
        if len(sim_values) == 0 or len(real_values) == 0:
            continue
        row = {
            "feature_set_name": feature_set_name,
            "feature": feature,
            "age_bin": str(age_bin),
            "age_bin_mid": float(age_bin.mid),
            "n_sim": len(sim_values),
            "n_real": len(real_values),
            "mean_sim": float(np.mean(sim_values)),
            "mean_real": float(np.mean(real_values)),
            "mean_difference": float(np.mean(sim_values) - np.mean(real_values)),
            "std_sim": float(np.std(sim_values, ddof=1)),
            "std_real": float(np.std(real_values, ddof=1)),
            "std_ratio_sim_to_real": float(np.std(sim_values, ddof=1) / np.std(real_values, ddof=1)),
            "q05_sim": float(np.quantile(sim_values, 0.05)),
            "q05_real": float(np.quantile(real_values, 0.05)),
            "q50_sim": float(np.quantile(sim_values, 0.50)),
            "q50_real": float(np.quantile(real_values, 0.50)),
            "q95_sim": float(np.quantile(sim_values, 0.95)),
            "q95_real": float(np.quantile(real_values, 0.95)),
        }
        for tail_name, (operator, threshold) in TAIL_DEFINITIONS.get(feature, {}).items():
            sim_tail = tail_rate(sim_values, operator, threshold)
            real_tail = tail_rate(real_values, operator, threshold)
            row[f"tail_{tail_name}_sim"] = sim_tail
            row[f"tail_{tail_name}_real"] = real_tail
            row[f"tail_{tail_name}_difference"] = sim_tail - real_tail
        rows.append(row)
    return rows


def compute_joint_dependence_metrics(feature_set_name, sim_frame, real_frame):
    if not {"bmi", "sbp"}.issubset(sim_frame.columns) or not {"bmi", "sbp"}.issubset(real_frame.columns):
        return []

    rows = []
    comparisons = [("overall", None)]
    sim_binned = sim_frame.copy()
    real_binned = real_frame.copy()
    sim_binned["age_bin"] = pd.cut(sim_binned["biological_age"], bins=AGE_BINS, include_lowest=True)
    real_binned["age_bin"] = pd.cut(real_binned["biological_age"], bins=AGE_BINS, include_lowest=True)
    for age_bin in sorted(set(sim_binned["age_bin"].dropna()) | set(real_binned["age_bin"].dropna())):
        comparisons.append((str(age_bin), age_bin))

    for label, age_bin in comparisons:
        if age_bin is None:
            sim = sim_frame[["bmi", "sbp"]].dropna()
            real = real_frame[["bmi", "sbp"]].dropna()
            age_bin_mid = np.nan
        else:
            sim = sim_binned.loc[sim_binned["age_bin"] == age_bin, ["bmi", "sbp"]].dropna()
            real = real_binned.loc[real_binned["age_bin"] == age_bin, ["bmi", "sbp"]].dropna()
            age_bin_mid = float(age_bin.mid)
        if len(sim) < 3 or len(real) < 3:
            continue
        corr_sim = float(sim["bmi"].corr(sim["sbp"]))
        corr_real = float(real["bmi"].corr(real["sbp"]))
        slope_sim = float(np.polyfit(sim["bmi"], sim["sbp"], deg=1)[0])
        slope_real = float(np.polyfit(real["bmi"], real["sbp"], deg=1)[0])
        rows.append(
            {
                "feature_set_name": feature_set_name,
                "age_bin": label,
                "age_bin_mid": age_bin_mid,
                "n_sim": len(sim),
                "n_real": len(real),
                "corr_bmi_sbp_sim": corr_sim,
                "corr_bmi_sbp_real": corr_real,
                "corr_difference": corr_sim - corr_real,
                "slope_sbp_on_bmi_sim": slope_sim,
                "slope_sbp_on_bmi_real": slope_real,
                "slope_difference": slope_sim - slope_real,
            }
        )
    return rows


def plot_shape_calibration(feature_set_name, sim_frame, real_frame, features):
    if not features:
        return
    fig, axes = plt.subplots(len(features), 2, figsize=(11, 3.7 * len(features)), squeeze=False)
    probs = np.linspace(0.01, 0.99, 99)
    for row_index, feature in enumerate(features):
        sim_values = numeric_clean(sim_frame, feature)
        real_values = numeric_clean(real_frame, feature)
        ax_hist, ax_qq = axes[row_index]
        sns.histplot(sim_values, stat="density", element="step", fill=False, label="synthetic", ax=ax_hist)
        sns.histplot(real_values, stat="density", element="step", fill=False, label="real", ax=ax_hist)
        ax_hist.set_title(f"{feature}: marginal shape")
        ax_hist.set_xlabel(feature)
        ax_hist.set_ylabel("Density")
        ax_hist.legend(title="domain")

        sim_q = np.quantile(sim_values, probs)
        real_q = np.quantile(real_values, probs)
        ax_qq.scatter(real_q, sim_q, s=12, alpha=0.8)
        min_q = min(real_q.min(), sim_q.min())
        max_q = max(real_q.max(), sim_q.max())
        ax_qq.plot([min_q, max_q], [min_q, max_q], linestyle="--", color="black", alpha=0.7)
        ax_qq.set_title(f"{feature}: synthetic vs real quantiles")
        ax_qq.set_xlabel("Real quantile")
        ax_qq.set_ylabel("Synthetic quantile")
        for ax in [ax_hist, ax_qq]:
            sns.despine(ax=ax)
    fig.suptitle(f"Continuous shape calibration: {feature_set_name}", y=1.01)
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / f"calibration_shape_continuous_{feature_set_name}.png", dpi=160, bbox_inches="tight")
    plt.close(fig)


def plot_age_conditional_calibration(feature_set_name, age_metrics, features):
    subset = age_metrics[age_metrics["feature_set_name"] == feature_set_name]
    if subset.empty:
        return
    fig, axes = plt.subplots(len(features), 2, figsize=(12, 3.7 * len(features)), squeeze=False)
    for row_index, feature in enumerate(features):
        feature_subset = subset[subset["feature"] == feature].sort_values("age_bin_mid")
        if feature_subset.empty:
            continue
        ax_mean, ax_sd = axes[row_index]
        ax_mean.plot(feature_subset["age_bin_mid"], feature_subset["mean_sim"], marker="o", label="synthetic")
        ax_mean.plot(feature_subset["age_bin_mid"], feature_subset["mean_real"], marker="o", label="real")
        ax_mean.fill_between(feature_subset["age_bin_mid"], feature_subset["q05_sim"], feature_subset["q95_sim"], alpha=0.15, label="synthetic q05-q95")
        ax_mean.fill_between(feature_subset["age_bin_mid"], feature_subset["q05_real"], feature_subset["q95_real"], alpha=0.15, label="real q05-q95")
        ax_mean.set_title(f"{feature}: age-conditional mean/tails")
        ax_mean.set_xlabel("Biological age bin midpoint")
        ax_mean.set_ylabel(feature)
        ax_mean.legend(fontsize=8)

        ax_sd.plot(feature_subset["age_bin_mid"], feature_subset["std_sim"], marker="o", label="synthetic")
        ax_sd.plot(feature_subset["age_bin_mid"], feature_subset["std_real"], marker="o", label="real")
        ax_sd.set_title(f"{feature}: age-conditional SD")
        ax_sd.set_xlabel("Biological age bin midpoint")
        ax_sd.set_ylabel("SD")
        ax_sd.legend(fontsize=8)
        for ax in [ax_mean, ax_sd]:
            sns.despine(ax=ax)
    fig.suptitle(f"Age-conditional continuous calibration: {feature_set_name}", y=1.01)
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / f"calibration_age_conditional_continuous_{feature_set_name}.png", dpi=160, bbox_inches="tight")
    plt.close(fig)


def plot_joint_dependence_calibration(feature_set_name, sim_frame, real_frame, joint_metrics):
    if not {"bmi", "sbp"}.issubset(sim_frame.columns) or not {"bmi", "sbp"}.issubset(real_frame.columns):
        return
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    rng = np.random.default_rng(SEED)
    n = min(1200, len(sim_frame), len(real_frame))
    sim_sample = sim_frame.sample(n=n, random_state=int(rng.integers(0, 1_000_000)))[["bmi", "sbp"]].assign(domain="synthetic")
    real_sample = real_frame.sample(n=n, random_state=int(rng.integers(0, 1_000_000)))[["bmi", "sbp"]].assign(domain="real")
    scatter_frame = pd.concat([sim_sample, real_sample], ignore_index=True)
    sns.scatterplot(data=scatter_frame, x="bmi", y="sbp", hue="domain", s=10, alpha=0.35, ax=axes[0])
    axes[0].set_title("BMI-SBP joint distribution")

    subset = joint_metrics[
        (joint_metrics["feature_set_name"] == feature_set_name)
        & (joint_metrics["age_bin"] != "overall")
    ].sort_values("age_bin_mid")
    axes[1].plot(subset["age_bin_mid"], subset["corr_bmi_sbp_sim"], marker="o", label="synthetic")
    axes[1].plot(subset["age_bin_mid"], subset["corr_bmi_sbp_real"], marker="o", label="real")
    axes[1].axhline(0, linestyle="--", color="black", alpha=0.5)
    axes[1].set_title("BMI-SBP correlation by age bin")
    axes[1].set_xlabel("Biological age bin midpoint")
    axes[1].set_ylabel("Correlation")
    axes[1].legend()
    for ax in axes:
        sns.despine(ax=ax)
    fig.suptitle(f"Joint dependence calibration: {feature_set_name}", y=1.01)
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / f"calibration_joint_dependence_bmi_sbp_{feature_set_name}.png", dpi=160, bbox_inches="tight")
    plt.close(fig)


shape_rows = []
age_rows = []
joint_rows = []
for feature_set_name, artifact in feature_set_artifacts.items():
    model = artifact["model"]
    features = target_continuous_features(model)
    sim_frame = artifact["synthetic_frame"]
    real_frame = artifact["real_frame"]
    for feature in features:
        shape_rows.append(compute_shape_metrics(feature_set_name, sim_frame, real_frame, feature))
        age_rows.extend(compute_age_conditional_metrics(feature_set_name, sim_frame, real_frame, feature))
    joint_rows.extend(compute_joint_dependence_metrics(feature_set_name, sim_frame, real_frame))

continuous_shape_calibration_metrics = pd.DataFrame(shape_rows)
age_conditional_continuous_calibration_metrics = pd.DataFrame(age_rows)
joint_dependence_calibration_metrics = pd.DataFrame(joint_rows)

continuous_shape_calibration_metrics.to_csv(OUTPUT_DIR / "continuous_shape_calibration_metrics.csv", index=False)
age_conditional_continuous_calibration_metrics.to_csv(OUTPUT_DIR / "age_conditional_continuous_calibration_metrics.csv", index=False)
joint_dependence_calibration_metrics.to_csv(OUTPUT_DIR / "joint_dependence_calibration_metrics.csv", index=False)

for feature_set_name, artifact in feature_set_artifacts.items():
    features = target_continuous_features(artifact["model"])
    plot_shape_calibration(feature_set_name, artifact["synthetic_frame"], artifact["real_frame"], features)
    plot_age_conditional_calibration(feature_set_name, age_conditional_continuous_calibration_metrics, features)
    plot_joint_dependence_calibration(
        feature_set_name,
        artifact["synthetic_frame"],
        artifact["real_frame"],
        joint_dependence_calibration_metrics,
    )

continuous_shape_calibration_metrics[["feature_set_name", "feature", "std_ratio_sim_to_real", "skew_difference", "q95_difference"]].round(3)


## Classifier Two-Sample Test

In [ ]:
def balanced_domain_sample(sim_frame, real_frame, feature_columns, seed):
    rng = np.random.default_rng(seed)
    n = min(len(sim_frame), len(real_frame), MAX_DOMAIN_CLASSIFIER_PER_CLASS)
    sim_sample = sim_frame.sample(n=n, random_state=int(rng.integers(0, 1_000_000)))
    real_sample = real_frame.sample(n=n, random_state=int(rng.integers(0, 1_000_000)))
    x = pd.concat([sim_sample[feature_columns], real_sample[feature_columns]], ignore_index=True)
    y = np.array([0] * n + [1] * n)
    return x, y


def evaluate_domain_classifier(feature_set_name, comparison_type, model_name, estimator, x, y, feature_columns, seed):
    x_train, x_test, y_train, y_test = train_test_split(
        x,
        y,
        test_size=DOMAIN_TEST_SIZE,
        random_state=seed,
        stratify=y,
    )
    estimator.fit(x_train, y_train)
    y_proba = estimator.predict_proba(x_test)[:, 1]
    y_pred = estimator.predict(x_test)
    auc = roc_auc_score(y_test, y_proba)
    accuracy = accuracy_score(y_test, y_pred)
    balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, label=f"AUC={auc:.2f}")
    ax.plot([0, 1], [0, 1], linestyle="--", color="black", alpha=0.7)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title(f"ROC: {feature_set_name}\n{comparison_type}, {model_name}")
    ax.legend()
    sns.despine(ax=ax)
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / f"roc_{feature_set_name}_{comparison_type}_{model_name}.png", dpi=160, bbox_inches="tight")
    plt.close(fig)

    if model_name == "logistic_regression":
        coefficients = estimator.named_steps["logisticregression"].coef_.reshape(-1)
        importance = np.abs(coefficients)
    elif model_name == "random_forest":
        importance = estimator.named_steps["randomforestclassifier"].feature_importances_
    else:
        importance = np.full(len(feature_columns), np.nan)

    importance_frame = pd.DataFrame(
        {
            "feature_set_name": feature_set_name,
            "comparison_type": comparison_type,
            "classifier": model_name,
            "feature": feature_columns,
            "importance": importance,
        }
    ).sort_values("importance", ascending=False)

    return (
        {
            "feature_set_name": feature_set_name,
            "comparison_type": comparison_type,
            "classifier": model_name,
            "roc_auc": float(auc),
            "accuracy": float(accuracy),
            "balanced_accuracy": float(balanced_accuracy),
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
            "n_test": int(len(y_test)),
        },
        importance_frame,
    )


classifier_rows = []
importance_tables = []
for feature_index, (feature_set_name, artifact) in enumerate(feature_set_artifacts.items()):
    feature_columns = artifact["model"]["columns"]
    comparisons = {
        "unconditional": (artifact["synthetic_frame"], artifact["real_frame"]),
    }
    sim_age_matched, real_age_matched = age_matched_frames(
        artifact["synthetic_frame"], artifact["real_frame"], seed=SEED + feature_index
    )
    if len(sim_age_matched) and len(real_age_matched):
        comparisons["age_matched"] = (sim_age_matched, real_age_matched)

    for comparison_index, (comparison_type, (sim_frame, real_frame)) in enumerate(comparisons.items()):
        x, y = balanced_domain_sample(
            sim_frame,
            real_frame,
            feature_columns,
            seed=CLASSIFIER_SEED_BASE + feature_index * 10 + comparison_index,
        )
        classifiers = {
            "logistic_regression": make_pipeline(
                SimpleImputer(strategy="median"),
                StandardScaler(),
                LogisticRegression(max_iter=2000, class_weight="balanced"),
            ),
            "random_forest": make_pipeline(
                SimpleImputer(strategy="median"),
                RandomForestClassifier(
                    n_estimators=300,
                    min_samples_leaf=5,
                    random_state=CLASSIFIER_SEED_BASE + feature_index,
                    class_weight="balanced_subsample",
                    n_jobs=-1,
                ),
            ),
        }
        for model_name, estimator in classifiers.items():
            row, importance_frame = evaluate_domain_classifier(
                feature_set_name,
                comparison_type,
                model_name,
                estimator,
                x,
                y,
                feature_columns,
                seed=CLASSIFIER_SEED_BASE + feature_index * 100 + comparison_index,
            )
            classifier_rows.append(row)
            importance_tables.append(importance_frame)

classifier_two_sample_results = pd.DataFrame(classifier_rows)
classifier_feature_importances = pd.concat(importance_tables, ignore_index=True)
classifier_two_sample_results.to_csv(OUTPUT_DIR / "classifier_two_sample_results.csv", index=False)
classifier_feature_importances.to_csv(OUTPUT_DIR / "classifier_feature_importances.csv", index=False)
classifier_two_sample_results


## Domain Feature Importance And Marginal Plots

In [ ]:
def plot_top_domain_features(feature_set_name, comparison_type="unconditional"):
    subset_scores = classifier_two_sample_results[
        (classifier_two_sample_results["feature_set_name"] == feature_set_name)
        & (classifier_two_sample_results["comparison_type"] == comparison_type)
    ]
    if subset_scores.empty:
        return
    best_classifier = subset_scores.sort_values("roc_auc", ascending=False).iloc[0]["classifier"]
    subset_importance = classifier_feature_importances[
        (classifier_feature_importances["feature_set_name"] == feature_set_name)
        & (classifier_feature_importances["comparison_type"] == comparison_type)
        & (classifier_feature_importances["classifier"] == best_classifier)
    ].head(TOP_N_CLASSIFIER_FEATURES)

    fig, ax = plt.subplots(figsize=(7, 4))
    sns.barplot(data=subset_importance, x="importance", y="feature", ax=ax)
    ax.set_title(f"Top domain classifier features\n{feature_set_name}, {best_classifier}")
    sns.despine(ax=ax)
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / f"top_domain_classifier_features_{feature_set_name}.png", dpi=160, bbox_inches="tight")
    plt.close(fig)


def binary_proportion_frame(sim_frame, real_frame, feature):
    rows = []
    for domain, frame in [("synthetic", sim_frame), ("real", real_frame)]:
        values = pd.to_numeric(frame[feature], errors="coerce").dropna()
        counts = values.astype(int).value_counts(normalize=True)
        for value in [0, 1]:
            rows.append(
                {
                    "domain": domain,
                    "value": str(value),
                    "proportion": float(counts.get(value, 0.0)),
                }
            )
    return pd.DataFrame(rows)


def plot_marginal_mismatch(feature_set_name):
    artifact = feature_set_artifacts[feature_set_name]
    model = artifact["model"]
    top_features = feature_mismatch_metrics[
        (feature_mismatch_metrics["feature_set_name"] == feature_set_name)
        & (feature_mismatch_metrics["comparison_type"] == "unconditional")
    ].head(TOP_N_MISMATCH_FEATURES)["feature"].tolist()
    if not top_features:
        return

    n_cols = min(3, len(top_features))
    n_rows = int(np.ceil(len(top_features) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows), squeeze=False)
    axes_flat = axes.flat

    for ax, feature in zip(axes_flat, top_features):
        sim_values = pd.to_numeric(artifact["synthetic_frame"][feature], errors="coerce").dropna()
        real_values = pd.to_numeric(artifact["real_frame"][feature], errors="coerce").dropna()

        if feature in model["binary_columns"]:
            proportions = binary_proportion_frame(
                artifact["synthetic_frame"],
                artifact["real_frame"],
                feature,
            )
            sns.barplot(data=proportions, x="value", y="proportion", hue="domain", ax=ax)
            ax.set_xlabel(f"{feature} value")
            ax.set_ylabel("Proportion")
            ax.set_ylim(0, 1)
            ax.set_xticks([0, 1])
            ax.set_xticklabels(["0", "1"])
        else:
            sns.histplot(
                sim_values,
                stat="density",
                common_norm=False,
                element="step",
                fill=False,
                label="synthetic",
                ax=ax,
            )
            sns.histplot(
                real_values,
                stat="density",
                common_norm=False,
                element="step",
                fill=False,
                label="real",
                ax=ax,
            )
            ax.set_xlabel(feature)
            ax.set_ylabel("Density")
            ax.legend(title="domain")

        ax.set_title(feature)
        sns.despine(ax=ax)

    for ax in list(axes_flat)[len(top_features):]:
        ax.axis("off")

    fig.suptitle(f"Top marginal mismatches: {feature_set_name}", y=1.02)
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / f"marginal_mismatch_top_features_{feature_set_name}.png", dpi=160, bbox_inches="tight")
    plt.close(fig)


for feature_set_name in FEATURE_SET_NAMES:
    plot_top_domain_features(feature_set_name)
    plot_marginal_mismatch(feature_set_name)


## PCA And Optional UMAP Domain Overlap

In [ ]:
pca_rows = []
umap_notes = []
try:
    import umap
    umap_available = True
except Exception as exc:
    umap_available = False
    umap_notes.append(f"UMAP skipped: {type(exc).__name__}: {exc}")

for feature_index, (feature_set_name, artifact) in enumerate(feature_set_artifacts.items()):
    feature_columns = artifact["model"]["columns"]
    x, y = balanced_domain_sample(
        artifact["synthetic_frame"],
        artifact["real_frame"],
        feature_columns,
        seed=PCA_SEED_BASE + feature_index,
    )
    x_scaled = make_pipeline(SimpleImputer(strategy="median"), StandardScaler()).fit_transform(x)
    pca = PCA(n_components=2, random_state=PCA_SEED_BASE + feature_index)
    embedding = pca.fit_transform(x_scaled)
    pca_rows.append(
        {
            "feature_set_name": feature_set_name,
            "pc1_explained_variance_ratio": float(pca.explained_variance_ratio_[0]),
            "pc2_explained_variance_ratio": float(pca.explained_variance_ratio_[1]),
            "fit_note": "PCA fitted on combined standardized synthetic + real features.",
        }
    )
    pca_frame = pd.DataFrame(
        {
            "pc1": embedding[:, 0],
            "pc2": embedding[:, 1],
            "domain": np.where(y == 0, "synthetic", "real"),
        }
    )
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.scatterplot(data=pca_frame, x="pc1", y="pc2", hue="domain", s=10, alpha=0.45, ax=ax)
    ax.set_title(f"PCA domain overlap: {feature_set_name}")
    sns.despine(ax=ax)
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / f"pca_domain_overlap_{feature_set_name}.png", dpi=160, bbox_inches="tight")
    plt.close(fig)

    if RUN_UMAP and umap_available:
        reducer = umap.UMAP(n_components=2, random_state=PCA_SEED_BASE + feature_index)
        umap_embedding = reducer.fit_transform(x_scaled)
        umap_frame = pd.DataFrame(
            {
                "umap1": umap_embedding[:, 0],
                "umap2": umap_embedding[:, 1],
                "domain": np.where(y == 0, "synthetic", "real"),
            }
        )
        fig, ax = plt.subplots(figsize=(6, 5))
        sns.scatterplot(data=umap_frame, x="umap1", y="umap2", hue="domain", s=10, alpha=0.45, ax=ax)
        ax.set_title(f"UMAP domain overlap: {feature_set_name}")
        sns.despine(ax=ax)
        plt.tight_layout()
        fig.savefig(PLOTS_DIR / f"umap_domain_overlap_{feature_set_name}.png", dpi=160, bbox_inches="tight")
        plt.close(fig)

pca_explained_variance = pd.DataFrame(pca_rows)
pca_explained_variance.to_csv(OUTPUT_DIR / "pca_explained_variance.csv", index=False)
pca_explained_variance


## Recovery Metrics From Saved Outputs

In [ ]:
def classify_recovery(synthetic_r, real_r):
    if pd.notna(real_r) and real_r >= ACCEPTABLE_REAL_R:
        return "acceptable_real"
    if pd.notna(synthetic_r) and synthetic_r >= GOOD_SYNTHETIC_R and (pd.isna(real_r) or real_r < POOR_R):
        return "good_synthetic_poor_real"
    if (pd.isna(synthetic_r) or synthetic_r < POOR_R) and (pd.isna(real_r) or real_r < POOR_R):
        return "poor_synthetic_poor_real"
    return "mixed_or_intermediate"


number_of_features_dir = EXPERIMENT_DIR / "results" / "data_diagnostics" / "number_of_features"
root_recovery_path = number_of_features_dir / f"feature_set_batch_count_grid_{SIMULATOR_VARIANT}.csv"
recovery_candidates = sorted(
    number_of_features_dir.glob(f"*/feature_set_batch_count_grid_{SIMULATOR_VARIANT}.csv"),
    key=lambda path: path.stat().st_mtime,
)
recovery_source_path = root_recovery_path if root_recovery_path.exists() else (recovery_candidates[-1] if recovery_candidates else root_recovery_path)
recovery_notes = []

if recovery_source_path.exists():
    saved_recovery = pd.read_csv(recovery_source_path)
    saved_recovery = saved_recovery[saved_recovery["feature_set"].isin(FEATURE_SET_NAMES)].copy()
    per_parameter_recovery = saved_recovery.assign(
        parameter="biological_age",
        synthetic_r_j=lambda df: df["synthetic_r"],
        real_r_j=lambda df: df["real_r"],
        recovery_gap_j=lambda df: df["synthetic_r"] - df["real_r"],
        good_synthetic_threshold=GOOD_SYNTHETIC_R,
        poor_threshold=POOR_R,
        acceptable_real_threshold=ACCEPTABLE_REAL_R,
    )
    per_parameter_recovery["recovery_category"] = [
        classify_recovery(s, r)
        for s, r in zip(per_parameter_recovery["synthetic_r_j"], per_parameter_recovery["real_r_j"])
    ]
    per_parameter_recovery = per_parameter_recovery.sort_values("recovery_gap_j", ascending=False)
else:
    recovery_notes.append(
        f"No saved recovery result found at {recovery_source_path}. "
        "Run notebook 02 number-of-features diagnostic first, or set up explicit retraining."
    )
    per_parameter_recovery = pd.DataFrame(
        columns=[
            "feature_set",
            "batch_count",
            "num_training_sims",
            "parameter",
            "synthetic_r_j",
            "real_r_j",
            "recovery_gap_j",
            "recovery_category",
            "good_synthetic_threshold",
            "poor_threshold",
            "acceptable_real_threshold",
        ]
    )

per_parameter_recovery.to_csv(OUTPUT_DIR / "per_parameter_recovery.csv", index=False)
per_parameter_recovery.head(12)


## Summary Tables And Plots

In [ ]:
domain_summary = (
    classifier_two_sample_results[classifier_two_sample_results["comparison_type"] == "unconditional"]
    .sort_values("roc_auc", ascending=False)
    .groupby("feature_set_name", as_index=False)
    .first()
    .rename(
        columns={
            "roc_auc": "domain_classifier_auc",
            "accuracy": "domain_classifier_accuracy",
            "balanced_accuracy": "domain_classifier_balanced_accuracy",
            "classifier": "best_domain_classifier",
        }
    )
)

mismatch_summary = (
    feature_mismatch_metrics[feature_mismatch_metrics["comparison_type"] == "unconditional"]
    .groupby("feature_set_name", as_index=False)
    .agg(
        max_abs_standardized_mean_difference=("absolute_standardized_mean_difference", "max"),
        mean_abs_standardized_mean_difference=("absolute_standardized_mean_difference", "mean"),
        max_wasserstein_distance=("wasserstein_distance", "max"),
    )
)

corr_summary = (
    correlation_mismatch_metrics.groupby("feature_set_name", as_index=False)
    .agg(
        mean_absolute_correlation_difference=("mean_absolute_correlation_difference", "first"),
        max_absolute_correlation_difference=("max_absolute_correlation_difference", "first"),
    )
)

if not per_parameter_recovery.empty:
    recovery_summary = (
        per_parameter_recovery.sort_values("num_training_sims")
        .groupby("feature_set", as_index=False)
        .tail(1)
        .rename(
            columns={
                "feature_set": "feature_set_name",
                "num_training_sims": "training_simulations",
                "synthetic_r_j": "overall_synthetic_recovery_pearson_r",
                "real_r_j": "overall_real_recovery_pearson_r",
                "recovery_gap_j": "overall_recovery_gap",
            }
        )[
            [
                "feature_set_name",
                "training_simulations",
                "final_train_loss",
                "min_train_loss",
                "overall_synthetic_recovery_pearson_r",
                "overall_real_recovery_pearson_r",
                "overall_recovery_gap",
            ]
        ]
    )
else:
    recovery_summary = pd.DataFrame({"feature_set_name": FEATURE_SET_NAMES})

summary_metrics_by_feature_set = (
    feature_set_summary.merge(domain_summary, on="feature_set_name", how="left")
    .merge(mismatch_summary, on="feature_set_name", how="left")
    .merge(corr_summary, on="feature_set_name", how="left")
    .merge(recovery_summary, on="feature_set_name", how="left")
)
summary_metrics_by_feature_set.to_csv(OUTPUT_DIR / "summary_metrics_by_feature_set.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=summary_metrics_by_feature_set, x="feature_set_name", y="domain_classifier_auc", ax=ax)
ax.axhline(0.5, linestyle="--", color="black", alpha=0.7)
ax.axhline(0.7, linestyle=":", color="black", alpha=0.7)
ax.set_ylim(0, 1)
ax.set_xlabel("Feature set")
ax.set_ylabel("Best domain classifier ROC AUC")
ax.set_title("Domain classifier AUC versus feature set")
ax.tick_params(axis="x", rotation=25)
sns.despine(ax=ax)
plt.tight_layout()
fig.savefig(PLOTS_DIR / "domain_classifier_auc_vs_feature_set.png", dpi=160, bbox_inches="tight")
plt.close(fig)

if not per_parameter_recovery.empty:
    recovery_plot = summary_metrics_by_feature_set.dropna(subset=["overall_synthetic_recovery_pearson_r", "overall_real_recovery_pearson_r"])
    if not recovery_plot.empty:
        long_recovery = recovery_plot.melt(
            id_vars="feature_set_name",
            value_vars=["overall_synthetic_recovery_pearson_r", "overall_real_recovery_pearson_r"],
            var_name="evaluation",
            value_name="pearson_r",
        )
        fig, ax = plt.subplots(figsize=(8, 4))
        sns.barplot(data=long_recovery, x="feature_set_name", y="pearson_r", hue="evaluation", ax=ax)
        ax.set_ylim(-0.05, 1.0)
        ax.set_xlabel("Feature set")
        ax.set_ylabel("Pearson r")
        ax.set_title("Recovery versus feature set")
        ax.tick_params(axis="x", rotation=25)
        sns.despine(ax=ax)
        plt.tight_layout()
        fig.savefig(PLOTS_DIR / "recovery_vs_feature_set.png", dpi=160, bbox_inches="tight")
        plt.close(fig)

        fig, ax = plt.subplots(figsize=(8, 4))
        sns.barplot(data=recovery_plot, x="feature_set_name", y="overall_recovery_gap", ax=ax)
        ax.axhline(0, linestyle="--", color="black", alpha=0.7)
        ax.set_xlabel("Feature set")
        ax.set_ylabel("Synthetic r - real r")
        ax.set_title("Recovery gap versus feature set")
        ax.tick_params(axis="x", rotation=25)
        sns.despine(ax=ax)
        plt.tight_layout()
        fig.savefig(PLOTS_DIR / "recovery_gap_vs_feature_set.png", dpi=160, bbox_inches="tight")
        plt.close(fig)

    for feature_set_name in FEATURE_SET_NAMES:
        subset = per_parameter_recovery[per_parameter_recovery["feature_set"] == feature_set_name]
        if subset.empty:
            continue
        latest_subset = subset.sort_values("num_training_sims").tail(1)
        long_subset = latest_subset.melt(
            id_vars=["parameter", "recovery_category"],
            value_vars=["synthetic_r_j", "real_r_j"],
            var_name="evaluation",
            value_name="pearson_r",
        )
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.barplot(data=long_subset, x="parameter", y="pearson_r", hue="evaluation", ax=ax)
        ax.set_ylim(-0.05, 1.0)
        ax.set_title(f"Per-parameter recovery: {feature_set_name}")
        sns.despine(ax=ax)
        plt.tight_layout()
        fig.savefig(PLOTS_DIR / f"per_parameter_recovery_{feature_set_name}.png", dpi=160, bbox_inches="tight")
        plt.close(fig)

        fig, ax = plt.subplots(figsize=(6, 4))
        sns.barplot(data=latest_subset, x="parameter", y="recovery_gap_j", hue="recovery_category", dodge=False, ax=ax)
        ax.axhline(0, linestyle="--", color="black", alpha=0.7)
        ax.set_title(f"Recovery gap: {feature_set_name}")
        sns.despine(ax=ax)
        plt.tight_layout()
        fig.savefig(PLOTS_DIR / f"recovery_gap_per_parameter_{feature_set_name}.png", dpi=160, bbox_inches="tight")
        plt.close(fig)

summary_metrics_by_feature_set


## Diagnostic Report

In [ ]:
def table_block(df, columns=None, n=12):
    if df.empty:
        return "No rows available."
    if columns is not None:
        columns = [col for col in columns if col in df.columns]
        df = df[columns]
    return "```text\n" + df.head(n).to_string(index=False) + "\n```"


top_domain_features = (
    classifier_feature_importances[
        (classifier_feature_importances["comparison_type"] == "unconditional")
    ]
    .sort_values(["feature_set_name", "importance"], ascending=[True, False])
    .groupby("feature_set_name")
    .head(5)
)

good_synthetic_poor_real = per_parameter_recovery[
    per_parameter_recovery["recovery_category"] == "good_synthetic_poor_real"
]
poor_synthetic_poor_real = per_parameter_recovery[
    per_parameter_recovery["recovery_category"] == "poor_synthetic_poor_real"
]
acceptable_real = per_parameter_recovery[
    per_parameter_recovery["recovery_category"] == "acceptable_real"
]

report_lines = [
    "# Synthetic-Real Feature Quality Diagnostic",
    "",
    f"Generated: {datetime.now().isoformat(timespec='seconds')}",
    f"Output directory: `{OUTPUT_DIR}`",
    "",
    "## 1. Executive Summary",
    "",
    "This diagnostic compares simulator-generated observed features against held-out baseline observed features for each configured feature set. It combines per-feature mismatch metrics, correlation mismatch, classifier two-sample tests, PCA overlap plots, and saved recovery metrics when available.",
    "",
]
if recovery_notes:
    report_lines.extend(["Recovery note:", *[f"- {note}" for note in recovery_notes], ""])
if umap_notes:
    report_lines.extend(["Optional diagnostic note:", *[f"- {note}" for note in umap_notes], ""])

report_lines.extend(
    [
        "## 2. Feature-Set Comparison Table",
        table_block(
            summary_metrics_by_feature_set,
            [
                "feature_set_name",
                "number_of_features",
                "domain_classifier_auc",
                "domain_classifier_accuracy",
                "domain_classifier_balanced_accuracy",
                "overall_synthetic_recovery_pearson_r",
                "overall_real_recovery_pearson_r",
                "overall_recovery_gap",
            ],
            n=20,
        ),
        "",
        "## 3. Synthetic-Real Mismatch Summary",
        table_block(
            feature_mismatch_metrics,
            [
                "feature_set_name",
                "comparison_type",
                "feature",
                "absolute_standardized_mean_difference",
                "wasserstein_distance",
                "ks_statistic",
            ],
            n=20,
        ),
        "",
        "## 4. Continuous Calibration Targets: Shape, Age Structure, And Joint Dependence",
        "",
        "These tables focus on continuous features present in each feature set, including the KDM8 lab biomarkers when available.",
        "",
        "Shape calibration summary:",
        table_block(
            continuous_shape_calibration_metrics,
            [
                "feature_set_name",
                "feature",
                "std_ratio_sim_to_real",
                "skew_difference",
                "q05_difference",
                "q50_difference",
                "q95_difference",
            ],
            n=20,
        ),
        "",
        "Joint dependence summary:",
        table_block(
            joint_dependence_calibration_metrics[joint_dependence_calibration_metrics["age_bin"] == "overall"],
            [
                "feature_set_name",
                "corr_bmi_sbp_sim",
                "corr_bmi_sbp_real",
                "corr_difference",
                "slope_sbp_on_bmi_sim",
                "slope_sbp_on_bmi_real",
                "slope_difference",
            ],
            n=20,
        ),
        "",
        "## 5. Classifier Two-Sample Test Results",
        table_block(classifier_two_sample_results, n=20),
        "",
        "## 6. Top Features Distinguishing Synthetic From Real",
        table_block(top_domain_features, ["feature_set_name", "classifier", "feature", "importance"], n=30),
        "",
        "## 7. Parameter Recovery Table",
        table_block(per_parameter_recovery, n=30),
        "",
        "## 8. Parameters With Good Synthetic Recovery But Poor Real Recovery",
        table_block(good_synthetic_poor_real, n=30),
        "",
        "## 9. Parameters With Poor Synthetic And Poor Real Recovery",
        table_block(poor_synthetic_poor_real, n=30),
        "",
        "## 10. Parameters With Acceptable Real Recovery",
        table_block(acceptable_real, n=30),
        "",
        "## 11. Cross-Feature-Set Interpretation",
        "",
        "Use `domain_classifier_auc_vs_feature_set.png`, `recovery_vs_feature_set.png`, and `recovery_gap_vs_feature_set.png` together. A high domain AUC with high synthetic recovery and low real recovery is evidence for simulator-real mismatch. Low synthetic and real recovery suggests weak identifiability from the selected features.",
        "",
        "## 12. Recommended Next Actions",
        "",
        "- Inspect feature sets with high domain classifier AUC first.",
        "- Prioritize simulator calibration for features with large standardized mean differences and high classifier importance.",
        "- If recovery results are unavailable, run notebook `02_feature_set_data_diagnostics.ipynb` first and rerun this notebook.",
        "- If synthetic and real are hard to distinguish but real recovery remains poor, investigate label noise, weak feature identifiability, or model capacity rather than feature-domain mismatch.",
    ]
)

report_path = OUTPUT_DIR / "diagnostic_report.md"
report_path.write_text("\n".join(report_lines))
report_path
